# SCHISM boundary conditions and namelists

**Learning goals:** Assemble already-prepared data sources into SCHISM boundary-condition types, understand how tidal/ocean/wave choices affect the model contract, and configure the namelist sections that activate those inputs.

**Prerequisites:** Journey 4; shared SCHISM fixtures. Journey 4 demonstrates source processing; this lesson focuses on model assembly and controls.

**Execution contract:** This lesson is **configuration-only**. It generates small boundary/namelist artefacts but never executes SCHISM or requires MPI/Docker.

## Checkpoint

By the end, explain which input file each boundary or namelist setting activates, and distinguish generated configuration from a validated model simulation.

Previous: [journey_04_schism_forcing](../journey_04_schism_forcing/)

Next: [journey_06_schism_real_case](../journey_06_schism_real_case/)


In [ ]:
import sys
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "scripts" / "schism_case_data.py").is_file())
sys.path.insert(0, str(root))
from scripts.schism_case_data import ensure_schism_data

case = ensure_schism_data()
print("Fixture directory:", case)


## Why this matters: model assembly

**Without Rompy:** boundary semantics and namelist switches are often maintained in separate hand-edited files, making it easy for generated filenames, forcing modes, and runtime options to drift apart.

**With Rompy:** boundary objects and namelist sections are validated as one configuration, while Journey 4 keeps the source transformations inspectable. This lesson focuses on the assembly contract and its verification.


## From prepared data to boundary types

Journey 4 prepared individual sources. Here they are assembled into SCHISM boundary semantics: HYCOM elevation supplies time-varying `elev2D.th.nc`, the tidal atlas supplies harmonic elevation and velocity coefficients in `bctides.in`, and wave spectra supply `wavedata.nc` for WWM coupling. The boundary type determines which variables SCHISM expects and how it interprets them.


In [ ]:
from rompy.core.source import SourceFile
from rompy_schism.data import SCHISMDataBoundary
from rompy_schism.boundary_conditions import create_hybrid_boundary_config

elevation = SCHISMDataBoundary(
    id="elev2D", source=SourceFile(uri=case / "hycom.nc"),
    variables=["surf_el"], coords={"t": "time", "y": "ylat", "x": "xlon"},
)
boundary_conditions = create_hybrid_boundary_config(
    constituents=["M2", "S2", "N2"],
    tidal_database=case / "tides", tidal_model="OCEANUM-atlas",
    nodal_corrections=True, tidal_potential=True, cutoff_depth=50.0,
    elev_source=elevation,
)
print("Boundary setup:", boundary_conditions.setup_type)
print("Configured constituents:", ["M2", "S2", "N2"])
print("Boundary source:", elevation.source.uri)
print("Boundary contract: harmonic tides + external elevation; Journey 4 generates the files.")


## Namelist controls the model contract

SCHISM namelists are not an afterthought: they activate physics and tell SCHISM where to find generated inputs. `param.core` controls the simulation, `param.schout` selects diagnostics, and `wwminput` configures wave coupling. Rompy updates data-source references when generated Sflux, boundary, or wave files are attached.


In [ ]:
from rompy_schism.namelists import NML
from rompy_schism.namelists.param import Param, Core, Schout
from rompy_schism.namelists.wwminput import Wwminput, Engs, Proc

namelist = NML(
    param=Param(
        core=Core(dt=150.0, rnday=1.0),
        schout=Schout(iof_hydro__1=1, iof_hydro__16=1, iof_wwm__1=1),
    ),
    wwminput=Wwminput(engs=Engs(fricc=0.006), proc=Proc(deltc=150.0)),
)
print("Core controls:", namelist.param.core.model_dump(exclude_none=True))
print("Output controls:", namelist.param.schout.model_dump(exclude_none=True))
print("WWM controls:", namelist.wwminput.model_dump(exclude_none=True))
print("Interpretation: these settings configure the runtime contract; they do not prove model stability or scientific skill.")


## What this lesson does—and does not—validate

Rompy can generate and connect `bctides.in`, `elev2D.th.nc`, Sflux, and WWM input files. Structural checks establish that filenames, variables, dimensions, and options are present. They do **not** establish tidal datum correctness, atlas suitability, coordinate/sign conventions, forcing continuity, or model skill. Those remain required expert checks before a production simulation.


In [ ]:
import matplotlib.pyplot as plt
from rompy.core.data import DataBlob
from rompy_schism import SCHISMGrid

assembly_grid = SCHISMGrid(
    hgrid=DataBlob(source=case / "hgrid.gr3"),
    vgrid=DataBlob(source=case / "vgrid.in"),
    drag=1,
)
fig, ax = plt.subplots(figsize=(8, 4))
assembly_grid.plot(ax=ax)
ax.scatter(*assembly_grid.boundary_points(), s=4, c="red", label="boundary used by assembly")
ax.set_title("Boundary locations used by SCHISM assembly")
ax.legend(); plt.show()
